# 04 · Attention audit across backbones and the ensemble

Extends `03_attention_audit.ipynb` to DenseNet-121, EfficientNet-B3 and the calibrated ensemble, with the same protocol: Grad-CAM min-max normalised per image, bilinear upsampling to the model input, masks resized with nearest neighbour and binarised at 127, EP / pointing / IoU at 0.5.

- Gate: ResNet-50 at `layer4[-1].conv3` must reproduce `results/attention/gradcam_segmentation_overlap.json` before anything else is used.
- Block-output layers: ResNet-50 `layer4`, DenseNet-121 `features` (norm5), EfficientNet-B3 `features` (last Conv2dNormActivation). Target = class predicted by each backbone; EfficientNet-B3 at 300 × 300.
- Ensemble map: the three block-output maps for the class predicted by the calibrated system (TTA ensemble + clinical thresholds, from the saved arrays), resized to 224 × 224, weighted by validation BACC and min-max normalised again. No TTA in the maps.

Runs on Colab with GPU. Expects the same Drive layout as notebook 03 plus `densenet121_best.pth` and `efficientnet_b3_best.pth` in `colab/`. Clones the public repository for the TTA arrays, thresholds, lesion-overlap flags and the reference test BACC.

File resolution is strict: each input must exist exactly once under `colab/` (checkpoints only in `colab/` itself, never in subfolders). Cell 2 checks that every checkpoint reproduces its test BACC in `results/seed_42/results.json` and writes `preflight.json` with paths, sizes and SHA-256 of every input.

Outputs in `OUT_DIR`: one JSON per map with the record format of notebook 03, and `summary.json`.

In [ ]:
# 1 · Setup: Drive, split, labels, images, masks, the three checkpoints and the calibrated predictions.

import os, glob, json, csv, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision.models import resnet50, densenet121, efficientnet_b3
import albumentations as A
from albumentations.pytorch import ToTensorV2

DRIVE_ROOT = "/content/drive/MyDrive/ISIC2018_Task3"
COLAB_DIR = os.path.join(DRIVE_ROOT, "colab")        # reference run (read only)
MASKS_DIR = os.path.join(DRIVE_ROOT, "HAM10000_segmentations_lesion_tschandl")
OUT_DIR = os.path.join(COLAB_DIR, "attention_backbones")
REPO_DIR = "/content/skin-lesion-classifier-CNN"
REPO_URL = "https://github.com/daorre1202/skin-lesion-classifier-CNN.git"

CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}
PRIORITY = ['MEL', 'AKIEC']
IOU_THR = 0.5
SIZES = {'resnet50': 224, 'densenet121': 224, 'efficientnet_b3': 300}
ENS_SIZE = 224
# validation BACC of each backbone, results/seed_42/results.json (eq. 1)
VAL_BACC = {'resnet50': 0.8117, 'densenet121': 0.8055, 'efficientnet_b3': 0.7907}
W = {k: v / sum(VAL_BACC.values()) for k, v in VAL_BACC.items()}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', "GPU runtime required"


def _mount_drive():
    if not os.path.isdir("/content/drive"):
        from google.colab import drive
        drive.mount('/content/drive')
    assert os.path.isdir(DRIVE_ROOT), f"{DRIVE_ROOT} not found"


def _find_one(filename, root, required=True):
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not hits:
        if required:
            raise FileNotFoundError(f"'{filename}' not found under {root}")
        return None
    if len(hits) > 1:
        print(f"  [warn] several matches for '{filename}', using the first:")
        for h in hits:
            print(f"    {h}")
    return hits[0]


def _exactly_one(filename, root, recursive=True):
    # stop if missing or ambiguous; never pick a match silently
    pat = os.path.join(root, "**", filename) if recursive else os.path.join(root, filename)
    hits = sorted(glob.glob(pat, recursive=recursive))
    assert hits, f"'{filename}' not found under {root}"
    assert len(hits) == 1, f"'{filename}' found {len(hits)} times under {root}: {hits}"
    return hits[0]


def _load_label_map():
    label_map = {}
    for name in ["ISIC2018_Task3_Training_GroundTruth.csv",
                 "ISIC2018_Task3_Validation_GroundTruth.csv",
                 "ISIC2018_Task3_Test_GroundTruth.csv"]:
        path = _find_one(name, DRIVE_ROOT, required=False)
        if path is None:
            print(f"  [warn] {name} not found, skipping")
            continue
        with open(path, newline='') as f:
            for row in csv.DictReader(f):
                onehot = {c: float(row[c]) for c in CLASSES}
                label_map[row['image']] = max(onehot, key=onehot.get)
    assert label_map, "no ground-truth CSV found"
    return label_map


def _index_raw_images():
    # same rule as notebook 03: first occurrence per id, sorted
    per_id = {}
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        for path in glob.glob(os.path.join(DRIVE_ROOT, "**", ext), recursive=True):
            img_id = os.path.splitext(os.path.basename(path))[0]
            if img_id not in per_id:
                per_id[img_id] = path
    return dict(sorted(per_id.items()))


def _build_mask_index(masks_dir):
    idx = {}
    for p in glob.glob(os.path.join(masks_dir, "**", "*.png"), recursive=True):
        base = os.path.splitext(os.path.basename(p))[0]
        for suf in ("_segmentation", "_Segmentation", "_seg", "_mask", "_Mask"):
            if base.endswith(suf):
                base = base[: -len(suf)]
                break
        idx[base] = p
    return idx


def _load(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = state['state_dict'] if isinstance(state, dict) and 'state_dict' in state else state
    model.load_state_dict(sd)
    return model.to(device).eval()


def build(name, ckpt_path):
    if name == 'resnet50':
        m = resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    elif name == 'densenet121':
        m = densenet121(weights=None); m.classifier = nn.Linear(m.classifier.in_features, len(CLASSES))
    else:
        m = efficientnet_b3(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return _load(m, ckpt_path)


# layers hooked per backbone; 'conv3' is the layer of notebook 03, the rest are block outputs
LAYERS = {
    'resnet50': {'conv3': lambda m: m.layer4[-1].conv3, 'block': lambda m: m.layer4},
    'densenet121': {'block': lambda m: m.features},
    'efficientnet_b3': {'block': lambda m: m.features},
}


def gradcams(model, name, x, target=None):
    """One forward/backward; returns ({layer: cam in [0,1] at feature resolution}, pred)."""
    store = {}
    handles = []
    for key, get in LAYERS[name].items():
        def _fh(mod, i, o, key=key):
            store[key] = {'a': o.detach().cpu()}
            o.register_hook(lambda g, key=key: store[key].__setitem__('g', g.detach().cpu()))
        handles.append(get(model).register_forward_hook(_fh))
    model.zero_grad()
    out = model(x.unsqueeze(0).to(device))
    pred = int(out.argmax(1).item())
    out[0, pred if target is None else target].backward()
    for h in handles:
        h.remove()
    cams = {}
    for key, d in store.items():
        g = d['g'].squeeze(0).numpy(); a = d['a'].squeeze(0).numpy()
        cam = np.maximum(np.einsum('c,chw->hw', g.mean(axis=(1, 2)), a), 0)
        mn, mx = cam.min(), cam.max()
        if mx - mn > 1e-8:
            cam = (cam - mn) / (mx - mn)
        cams[key] = cam
    return cams, pred


def upsample_cam(cam, size):
    # same quantisation as notebook 03
    im = Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    return np.array(im).astype(np.float32) / 255.0


def load_mask(mask_path, size):
    m = np.array(Image.open(mask_path).convert('L').resize((size, size), Image.NEAREST))
    return m > 127


def overlap(cam, mask):
    eps = 1e-8
    ep = float((cam * mask).sum() / (cam.sum() + eps))
    peak = np.unravel_index(int(cam.argmax()), cam.shape)
    camb = cam >= IOU_THR
    union = np.logical_or(camb, mask).sum()
    iou = float(np.logical_and(camb, mask).sum() / (union + eps)) if union > 0 else float('nan')
    return ep, bool(mask[peak]), iou


_NORM = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
TF = {s: A.Compose([A.Resize(s, s), _NORM, ToTensorV2()]) for s in (224, 300)}

print("1) Drive"); _mount_drive()
os.makedirs(OUT_DIR, exist_ok=True)

print("2) repository arrays")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
probs = np.loadtxt(os.path.join(REPO_DIR, "results/seed_42/tta_sum_probs.csv"), delimiter=",")
labels = np.loadtxt(os.path.join(REPO_DIR, "results/seed_42/tta_labels.csv"), dtype=int)
thr = json.load(open(os.path.join(REPO_DIR, "results/seed_42/calibrated_thresholds.json")))
flags = pd.read_csv(os.path.join(REPO_DIR, "results/lesion_overlap/flags.csv"))
ens_pred = probs.argmax(1).copy()
for n in reversed(PRIORITY):
    ens_pred[probs[:, CLASSES.index(n)] > thr[n]] = CLASSES.index(n)
test_flags = flags[flags.split == 'test']
ENS_PRED = {r.image_id: CLASSES[ens_pred[r.position]] for r in test_flags.itertuples()}
ENS_TRUE = {r.image_id: CLASSES[labels[r.position]] for r in test_flags.itertuples()}
print(f"  [ok] calibrated predictions for {len(ENS_PRED)} test images")

print("3) split, labels, images, masks")
SPLIT_PATH = _exactly_one("split_assignment.json", COLAB_DIR)
with open(SPLIT_PATH) as f:
    split_assignment = json.load(f)
test_ids_all = sorted(i for i, sp in split_assignment.items() if sp == 'test')
from collections import Counter
_counts = dict(Counter(split_assignment.values()))
assert _counts == {'train': 7030, 'val': 2342, 'test': 2348}, _counts
print(f"  [ok] {SPLIT_PATH}: {_counts}")
label_map = _load_label_map()
id2path = _index_raw_images()
mask_index = _build_mask_index(MASKS_DIR)
test_records = [(i, id2path[i], label_map[i]) for i in test_ids_all
                if i in id2path and i in label_map and i in mask_index]
print(f"  [ok] {len(test_ids_all)} test ids, {len(test_records)} with image, label and mask")
# the repository arrays must refer to the same test images and labels as this split
assert set(ENS_TRUE) == set(test_ids_all), "test ids differ from the repository arrays"
assert all(ENS_TRUE[i] == label_map[i] for i in test_ids_all), "label mismatch with the repository arrays"
print("  [ok] repository arrays match the split and the labels on all 2348 test images")

print("4) checkpoints")
MODELS, CKPTS = {}, {}
for name in ('resnet50', 'densenet121', 'efficientnet_b3'):
    # only colab/ itself: subfolders hold other runs (e.g. attention supervision)
    ck = _exactly_one(f"{name}_best.pth", COLAB_DIR, recursive=False)
    CKPTS[name] = ck
    MODELS[name] = build(name, ck)
    print(f"  [ok] {name}: {ck}")

print("5) reference records")
REF_PATH = _exactly_one("gradcam_segmentation_overlap.json", COLAB_DIR)
_ref = json.load(open(REF_PATH))
assert _ref['config']['model'] == 'resnet50_seed42' and _ref['config']['gradcam_target'] == 'pred', _ref['config']
REF = {r['id']: r for r in _ref['records']}
assert len(REF) == len(test_records) == 1991, (len(REF), len(test_records))
print(f"  [ok] {len(REF)} reference records: {REF_PATH}")

1) Drive
Mounted at /content/drive
2) repository arrays
  [ok] calibrated predictions for 2348 test images
3) split, labels, images, masks
  [ok] /content/drive/MyDrive/ISIC2018_Task3/colab/split_assignment.json: {'train': 7030, 'val': 2342, 'test': 2348}
  [warn] several matches for 'ISIC2018_Task3_Training_GroundTruth.csv', using the first:
    /content/drive/MyDrive/ISIC2018_Task3/Groundtruth/ISIC2018_Task3_Training_GroundTruth.csv
    /content/drive/MyDrive/ISIC2018_Task3/TFG/ISIC2018_Task3_Training_GroundTruth.csv
  [warn] several matches for 'ISIC2018_Task3_Validation_GroundTruth.csv', using the first:
    /content/drive/MyDrive/ISIC2018_Task3/Groundtruth/ISIC2018_Task3_Validation_GroundTruth.csv
    /content/drive/MyDrive/ISIC2018_Task3/TFG/ISIC2018_Task3_Validation_GroundTruth.csv
  [warn] several matches for 'ISIC2018_Task3_Test_GroundTruth.csv', using the first:
    /content/drive/MyDrive/ISIC2018_Task3/Groundtruth/ISIC2018_Task3_Test_GroundTruth.csv
    /content/drive/MyDriv

In [ ]:
# 2 · Check: each checkpoint reproduces its test BACC in results/seed_42/results.json,
# then a manifest of every input. Single-model eval without TTA, same transform as the pipeline.

import hashlib
from sklearn.metrics import balanced_accuracy_score

REF_BACC = json.load(open(os.path.join(REPO_DIR, "results/seed_42/results.json")))['models']
all_test = [(i, id2path[i], label_map[i]) for i in test_ids_all if i in id2path]
assert len(all_test) == 2348, f"{len(all_test)} test images resolved, expected 2348"
y_true = [CLASSES.index(c) for _, _, c in all_test]
preds = {n: [] for n in MODELS}
BATCH = 32
with torch.no_grad():
    for k in range(0, len(all_test), BATCH):
        imgs = [np.array(Image.open(p).convert('RGB')) for _, p, _ in all_test[k:k + BATCH]]
        for n, m in MODELS.items():
            xb = torch.stack([TF[SIZES[n]](image=im)['image'] for im in imgs]).to(device)
            preds[n] += m(xb).argmax(1).cpu().tolist()
        if (k // BATCH) % 20 == 0:
            print(f"  {k}/{len(all_test)}")
ok = True
for n in MODELS:
    b = balanced_accuracy_score(y_true, preds[n])
    ref = REF_BACC[n]['test_bacc']
    good = abs(b - ref) < 5e-4
    ok &= good
    print(f"  {n:16s} test BACC {b:.4f}  results.json {ref:.4f}  {'ok' if good else 'MISMATCH'}")
assert ok, "a checkpoint does not reproduce results.json: wrong file, stop here"


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


inputs = {**{f"checkpoint_{n}": p for n, p in CKPTS.items()},
          'split_assignment': SPLIT_PATH, 'reference_records': REF_PATH}
manifest = {k: {'path': p, 'bytes': os.path.getsize(p), 'sha256': _sha256(p)} for k, p in inputs.items()}
manifest['repo_commit'] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                         capture_output=True, text=True).stdout.strip()
manifest['masks'] = {'dir': MASKS_DIR, 'n_files': len(mask_index)}
json.dump(manifest, open(os.path.join(OUT_DIR, 'preflight.json'), 'w'), indent=1)
for k, v in manifest.items():
    print(f"  {k}: {v}")
print("[ok] preflight passed")

  0/2348
  640/2348
  1280/2348
  1920/2348
  resnet50         test BACC 0.8135  results.json 0.8135  ok
  densenet121      test BACC 0.8017  results.json 0.8017  ok
  efficientnet_b3  test BACC 0.8007  results.json 0.8007  ok
  checkpoint_resnet50: {'path': '/content/drive/MyDrive/ISIC2018_Task3/colab/resnet50_best.pth', 'bytes': 94409443, 'sha256': 'ae5b2df512ecb277cef9a982029aea2ca5ec7da32e03d84b9c2f8ac3860fa552'}
  checkpoint_densenet121: {'path': '/content/drive/MyDrive/ISIC2018_Task3/colab/densenet121_best.pth', 'bytes': 28455702, 'sha256': '51b88413c04f7d3ce5e168a46b11b454a12fdca9cecaabedbb6c3bd92c33bbce'}
  checkpoint_efficientnet_b3: {'path': '/content/drive/MyDrive/ISIC2018_Task3/colab/efficientnet_b3_best.pth', 'bytes': 43390839, 'sha256': 'f5fc2fd21830b863464f2e062acb61b7b5388c592554935b4a8d510700dd1197'}
  split_assignment: {'path': '/content/drive/MyDrive/ISIC2018_Task3/colab/split_assignment.json', 'bytes': 285968, 'sha256': '378b091ca86b477ab2b08c2ed70dece92c721b33fd373

In [ ]:
# 3 · Gate: ResNet-50 at layer4[-1].conv3 must reproduce notebook 03 before running everything.

GATE_N = 200
diffs, flips = [], 0
for iid, path, cls in test_records[:GATE_N]:
    x = TF[224](image=np.array(Image.open(path).convert('RGB')))['image']
    cams, pred = gradcams(MODELS['resnet50'], 'resnet50', x)
    ep, _, _ = overlap(upsample_cam(cams['conv3'], 224), load_mask(mask_index[iid], 224))
    diffs.append(abs(ep - REF[iid]['energy_prop']))
    flips += CLASSES[pred] != REF[iid]['pred']
print(f"max |EP diff| = {max(diffs):.2e}  mean = {np.mean(diffs):.2e}  prediction changes = {flips}")
assert flips == 0 and max(diffs) < 1e-3, "does not reproduce notebook 03: stop and check before going on"
print("[ok] gate passed")

max |EP diff| = 2.98e-07  mean = 3.74e-08  prediction changes = 0
[ok] gate passed


In [4]:
# 4 · Full run. Resumable: progress saved to Drive every CHECKPOINT_EVERY images.

import time, sys

RESUME = True
CHECKPOINT_EVERY = 100
MAPS = ['resnet50_conv3', 'resnet50_layer4', 'densenet121', 'efficientnet_b3', 'ensemble']
ckpt_path = os.path.join(OUT_DIR, 'backbones_CHECKPOINT.json')

recs = {k: [] for k in MAPS}
done = set()
if RESUME and os.path.isfile(ckpt_path):
    recs = json.load(open(ckpt_path))
    done = {r['id'] for r in recs['ensemble']}
    print(f"[resume] {len(done)} done")


def rec(iid, cls, pred, ep, pg, iou, mask):
    return {'id': iid, 'true': cls, 'pred': pred, 'correct': bool(pred == cls),
            'energy_prop': ep, 'pointing_hit': pg, 'iou': iou, 'mask_frac': float(mask.mean())}


pending = [t for t in test_records if t[0] not in done]
t0 = time.time()
for k, (iid, path, cls) in enumerate(pending, start=1):
    img = np.array(Image.open(path).convert('RGB'))
    m224 = load_mask(mask_index[iid], 224)
    ens_cls = ENS_PRED[iid]
    ens_maps = []
    for name, model in MODELS.items():
        s = SIZES[name]
        x = TF[s](image=img)['image']
        msk = m224 if s == 224 else load_mask(mask_index[iid], s)
        cams, pred = gradcams(model, name, x)
        p = CLASSES[pred]
        if name == 'resnet50':
            recs['resnet50_conv3'].append(rec(iid, cls, p, *overlap(upsample_cam(cams['conv3'], s), msk), msk))
            recs['resnet50_layer4'].append(rec(iid, cls, p, *overlap(upsample_cam(cams['block'], s), msk), msk))
        else:
            recs[name].append(rec(iid, cls, p, *overlap(upsample_cam(cams['block'], s), msk), msk))
        # map for the class of the calibrated system; reuse the pass when it is the same class
        if p != ens_cls:
            cams, _ = gradcams(model, name, x, target=CLASSES.index(ens_cls))
        ens_maps.append(W[name] * upsample_cam(cams['block'], ENS_SIZE))
    ens = np.sum(ens_maps, axis=0)
    mn, mx = ens.min(), ens.max()
    if mx - mn > 1e-8:
        ens = (ens - mn) / (mx - mn)
    recs['ensemble'].append(rec(iid, cls, ens_cls, *overlap(ens, m224), m224))
    if k % CHECKPOINT_EVERY == 0 or k == len(pending):
        json.dump(recs, open(ckpt_path, 'w'))
        print(f"  {k}/{len(pending)} | {time.time() - t0:.0f}s"); sys.stdout.flush()

# full-run check against notebook 03 on every image
d = [abs(r['energy_prop'] - REF[r['id']]['energy_prop']) for r in recs['resnet50_conv3']]
print(f"conv3 vs notebook 03 on {len(d)} images: max |EP diff| = {max(d):.2e}")
assert max(d) < 1e-3

CONFIG = {
    'resnet50_conv3': 'ResNet-50 layer4[-1].conv3, own prediction (reproduces notebook 03)',
    'resnet50_layer4': 'ResNet-50 layer4 output, own prediction',
    'densenet121': 'DenseNet-121 features (norm5) output, own prediction',
    'efficientnet_b3': 'EfficientNet-B3 features output, own prediction, 300x300',
    'ensemble': 'block-output maps of the three backbones for the class of the calibrated system '
                '(TTA ensemble + clinical thresholds), 224x224, weighted by validation BACC, '
                'min-max normalised; no TTA in the maps',
}
for k in MAPS:
    json.dump({'config': {'map': CONFIG[k], 'iou_threshold': IOU_THR,
                          'mask_source': 'tschandl/ham10000-lesion-segmentations'},
               'records': recs[k]}, open(os.path.join(OUT_DIR, f'{k}.json'), 'w'), indent=1)
print("[ok] per-map JSON written to", OUT_DIR)

  100/1991 | 21s
  200/1991 | 40s
  300/1991 | 101s
  400/1991 | 161s
  500/1991 | 221s
  600/1991 | 284s
  700/1991 | 345s
  800/1991 | 408s
  900/1991 | 469s
  1000/1991 | 530s
  1100/1991 | 590s
  1200/1991 | 649s
  1300/1991 | 708s
  1400/1991 | 767s
  1500/1991 | 826s
  1600/1991 | 886s
  1700/1991 | 946s
  1800/1991 | 1008s
  1900/1991 | 1070s
  1991/1991 | 1125s
conv3 vs notebook 03 on 1991 images: max |EP diff| = 1.04e-04
[ok] per-map JSON written to /content/drive/MyDrive/ISIC2018_Task3/colab/attention_backbones


In [5]:
# 5 · Summary per map: EP by subgroup, effect sizes, error analysis by class. CPU only.

from scipy.stats import mannwhitneyu

rng = np.random.default_rng(42)


def r_rb(a, b):
    return 2 * mannwhitneyu(a, b, alternative='two-sided').statistic / (len(a) * len(b)) - 1


def cmp(a, b, n_boot=2000):
    a, b = np.asarray(a), np.asarray(b)
    bs = [r_rb(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(n_boot)]
    return {'n': [len(a), len(b)], 'mean': [round(float(a.mean()), 3), round(float(b.mean()), 3)],
            'r_rb': round(float(r_rb(a, b)), 3),
            'r_rb_ci95': [round(float(v), 3) for v in np.percentile(bs, [2.5, 97.5])],
            'p': float(f"{mannwhitneyu(a, b, alternative='two-sided').pvalue:.2g}")}


no_mate = set(test_flags[(test_flags.has_lesion_id == 1) & (test_flags.overlap == 0)].image_id)
summary = {}
for k in MAPS:
    df = pd.DataFrame(recs[k])
    out = {}
    for tag, s in (('all', df), ('no_lesion_mate', df[df.id.isin(no_mate)])):
        mal = s.true.isin(MALIGNANT)
        o = {'n': len(s), 'ep_overall': round(float(s.energy_prop.mean()), 3),
             'ep_malignant': round(float(s.energy_prop[mal].mean()), 3),
             'ep_benign': round(float(s.energy_prop[~mal].mean()), 3),
             'pg_malignant': round(float(s.pointing_hit[mal].mean()), 3),
             'pg_benign': round(float(s.pointing_hit[~mal].mean()), 3),
             'malignant_vs_benign': cmp(s.energy_prop[mal], s.energy_prop[~mal])}
        for c in ('MEL', 'NV'):
            t = s[s.true == c]
            if (~t.correct).sum() >= 5 and t.correct.sum() >= 5:
                o[f'{c}_error_vs_correct'] = cmp(t.energy_prop[~t.correct], t.energy_prop[t.correct])
        out[tag] = o
    summary[k] = out
    a = out['all']
    print(f"{k:16s} EP {a['ep_overall']:.3f} | mal {a['ep_malignant']:.3f} ben {a['ep_benign']:.3f} "
          f"r {a['malignant_vs_benign']['r_rb']:+.2f} | MEL err r {a.get('MEL_error_vs_correct', {}).get('r_rb', float('nan')):+.2f} "
          f"| NV err r {a.get('NV_error_vs_correct', {}).get('r_rb', float('nan')):+.2f}")
json.dump(summary, open(os.path.join(OUT_DIR, 'summary.json'), 'w'), indent=1)
print("[ok]", os.path.join(OUT_DIR, 'summary.json'))

resnet50_conv3   EP 0.507 | mal 0.632 ben 0.476 r +0.37 | MEL err r -0.48 | NV err r +0.41
resnet50_layer4  EP 0.469 | mal 0.605 ben 0.436 r +0.41 | MEL err r -0.46 | NV err r +0.47
densenet121      EP 0.438 | mal 0.563 ben 0.407 r +0.40 | MEL err r -0.55 | NV err r +0.32
efficientnet_b3  EP 0.748 | mal 0.818 ben 0.731 r +0.33 | MEL err r -0.68 | NV err r +0.40
ensemble         EP 0.503 | mal 0.620 ben 0.475 r +0.38 | MEL err r -0.49 | NV err r +0.35
[ok] /content/drive/MyDrive/ISIC2018_Task3/colab/attention_backbones/summary.json
